In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

## Data Collection

In [ ]:
funds = ["FSPTX", "FSELX", "FSCSX", "FSPHX", "FBIOX", "FSMEX", "FPHAX", "FIDSX", "FSRBX", "FSPCX", "FSENX", "FSDAX", "FSRFX", "FSCPX", "FSAVX", "FSRPX", "FDFAX", "FSTCX", "FWRLX"]
prices = yf.download(tickers=funds + ["SPY", "XLK"], start="2015-01-01", end="2026-10-01", auto_adjust=False, progress=False)

In [ ]:
prices = prices["Adj Close"]
prices

In [ ]:
log_returns = np.log(prices).diff().dropna()
market_log_returns = log_returns["SPY"]
xlk_log_returns = log_returns["XLK"]
log_returns = log_returns[funds]
print(log_returns)

## Performace Summary

### Part 1

In [ ]:
mean_returns = log_returns.mean() * 252
volatilities = log_returns.std() * np.sqrt(252)
sharpe_ratio = mean_returns / volatilities

absolute = pd.DataFrame({
    "Mean Return": mean_returns,
    "Volatility": volatilities,
    "Sharpe Ratio": sharpe_ratio,
})

summary_abs = absolute.describe().loc[["min", "25%", "mean", "50%", "75%", "max"]]
summary_abs.index = ["Min", "Q1", "Mean", "Median", "Q3", "Max"]
summary_abs.round(4)

### Part 2

In [ ]:
market_mean_return = market_log_returns.mean() * 252
market_volatility = market_log_returns.std() * np.sqrt(252)

fund_metrics = pd.DataFrame({
    "Mean Return": mean_returns,
    "Volatility": volatilities,
})
plt.scatter(fund_metrics["Volatility"], fund_metrics["Mean Return"], label="Funds")
for t in fund_metrics.index:
    plt.annotate(t, (fund_metrics.loc[t, "Volatility"], fund_metrics.loc[t, "Mean Return"]),
                 fontsize=8, xytext=(3, 3), textcoords="offset points")
plt.scatter(market_volatility, market_mean_return, color="red", marker="o", label="Market (SPY)")
plt.xlabel("Volatility")
plt.ylabel("Mean Return")
plt.title("Fund Performance")
plt.grid(True)
plt.legend()
plt.show()

indices = fund_metrics["Mean Return"] > market_mean_return
outperforms_market = fund_metrics[indices].copy()
outperforms_market

market_sharpe_ratio = market_mean_return / market_volatility
print(market_sharpe_ratio)
outperforms_market["Sharpe Ratio"] = outperforms_market["Mean Return"] / outperforms_market["Volatility"]
outperforms_market

Only the three technology-oriented funds, Software (FSCSX, 14.5%), Technology (FSPTX, 21.3%) and Semiconductors (FSELX, 26.5%), outperform SPY's 12.9% mean return, but each does so with higher volatility (23.5%-34.3% vs. 17.6% for SPY).

### Part 3

#### (a)

In [ ]:
combined = log_returns.copy()
combined["SPY"] = market_log_returns
covariances = combined.cov() * 252
betas = (covariances["SPY"] / covariances.loc["SPY", "SPY"]).drop("SPY")

alphas = mean_returns - betas * market_mean_return
alphas

treynor = (mean_returns) / betas
treynor

tracking_error = log_returns.sub(market_log_returns, axis=0)
tracking_error = tracking_error.std() * np.sqrt(252)

information_ratio = (mean_returns - market_mean_return) / tracking_error
information_ratio

relative = pd.DataFrame({
    "Jensen's Alpha": alphas,
    "Beta": betas,
    "Treynor Ratio": treynor,
    "Tracking Error": tracking_error,
    "Information Ratio": information_ratio,
})

summary = relative.describe().loc[["min", "25%", "mean", "50%", "75%", "max"]]
summary.index = ["Min", "Q1", "Mean", "Median", "Q3", "Max"]
summary

(b)

In [ ]:
area = pd.Series({
    "FSPTX": "Technology", "FSELX": "Semiconductors", "FSCSX": "Software",
    "FSPHX": "Health Care", "FBIOX": "Biotechnology", "FSMEX": "Medical Technology",
    "FPHAX": "Pharmaceuticals", "FIDSX": "Financials", "FSRBX": "Banking",
    "FSPCX": "Insurance", "FSENX": "Energy", "FSDAX": "Aerospace/Defense",
    "FSRFX": "Transportation", "FSCPX": "Consumer Discretionary", "FSAVX": "Automotive",
    "FSRPX": "Retailing", "FDFAX": "Consumer Staples", "FSTCX": "Telecommunications",
    "FWRLX": "Wireless",
})

top5 = alphas.sort_values(ascending=False).index[:5]
top5_table = pd.concat([area[top5].rename("Area"), relative.loc[top5, ["Jensen's Alpha", "Beta"]], absolute.loc[top5]], axis=1)
top5_table["Mean Return Rank"] = mean_returns.rank(ascending=False)[top5].astype(int)
top5_table["Sharpe Rank"] = sharpe_ratio.rank(ascending=False)[top5].astype(int)
top5_table.round(4)

### 4.

#### (a)

In [ ]:
expense = pd.Series({
    "FSPTX": 0.61, "FSELX": 0.60, "FSCSX": 0.61, "FSPHX": 0.62, "FBIOX": 0.62,
    "FSMEX": 0.62, "FPHAX": 0.67, "FIDSX": 0.68, "FSRBX": 0.69, "FSPCX": 0.68,
    "FSENX": 0.65, "FSDAX": 0.63, "FSRFX": 0.68, "FSCPX": 0.69, "FSAVX": 0.78,
    "FSRPX": 0.63, "FDFAX": 0.68, "FSTCX": 0.74, "FWRLX": 0.71,
})

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

ax[0].scatter(expense[funds], mean_returns[funds])
ax[0].set_xlabel("Expense ratio (%)")
ax[0].set_ylabel("Annual mean return")
ax[0].set_title("Mean Return vs. Expense Ratio")
ax[0].grid(True)

ax[1].scatter(expense[funds], alphas[funds])
ax[1].axhline(0, color="gray", linewidth=0.8)
ax[1].set_xlabel("Expense ratio (%)")
ax[1].set_ylabel("Jensen's alpha (SPY)")
ax[1].set_title("Jensen's Alpha vs. Expense Ratio")
ax[1].grid(True)

for t in funds:
    ax[0].annotate(t, (expense[t], mean_returns[t]), fontsize=7, xytext=(3, 3), textcoords="offset points")
    ax[1].annotate(t, (expense[t], alphas[t]), fontsize=7, xytext=(3, 3), textcoords="offset points")

plt.tight_layout()
plt.savefig("fig_1_4.png", dpi=300)
plt.show()

#### (b)

In [ ]:
print("Corr(ER, Mean): ", round(expense[funds].corr(mean_returns[funds]), 4))
print("Corr(ER, Alpha):", round(expense[funds].corr(alphas[funds]), 4))

### 5.

#### (a)

In [ ]:
xlk_mean_return = xlk_log_returns.mean() * 252

returns_xlk = log_returns[funds].copy()
returns_xlk["XLK"] = xlk_log_returns
cov_xlk = returns_xlk.cov() * 252
betas_xlk = (cov_xlk["XLK"] / cov_xlk.loc["XLK", "XLK"]).drop("XLK")

alphas_xlk = mean_returns - betas_xlk * xlk_mean_return
treynor_xlk = mean_returns / betas_xlk
tracking_error_xlk = log_returns[funds].sub(xlk_log_returns, axis=0).std() * np.sqrt(252)
information_ratio_xlk = (mean_returns - xlk_mean_return) / tracking_error_xlk

relative_xlk = pd.DataFrame({
    "Jensen's Alpha": alphas_xlk,
    "Beta": betas_xlk,
    "Treynor Ratio": treynor_xlk,
    "Tracking Error": tracking_error_xlk,
    "Information Ratio": information_ratio_xlk,
})
summary_xlk = relative_xlk.describe().loc[["min", "25%", "mean", "50%", "75%", "max"]]
summary_xlk.index = ["Min", "Q1", "Mean", "Median", "Q3", "Max"]
display(summary_xlk.round(4))

#### (b)

In [ ]:
compare = pd.DataFrame({
    "Alpha_SPY": alphas[funds],
    "Rank_SPY": alphas[funds].rank(ascending=False).astype(int),
    "Alpha_XLK": alphas_xlk,
    "Rank_XLK": alphas_xlk.rank(ascending=False).astype(int),
}).sort_values("Rank_SPY")
display(compare.round(4))

### Mean-Variance Efficient Frontier

#### 1. Monthly Data and Split

In [ ]:
tickers_2 = ["FSPTX","FIDSX","FSCPX"]
prices_2 = yf.download(tickers=tickers_2, start="2015-01-01", end="2026-10-01", interval ="1mo", auto_adjust=False, progress=False)["Adj Close"][tickers_2]

log_returns_2 = np.log(prices_2).diff().dropna()

half = len(log_returns_2) // 2
r1 = log_returns_2.iloc[:half]
r2 = log_returns_2.iloc[half:]
print(len(r1), r1.index[0],r1.index[-1])
print(len(r2), r2.index[0],r2.index[-1])
returns_1 = r1
returns_2 = r2
print(returns_1)
print(returns_2)

#### 2. Mean Vectors

In [ ]:
mu1 = r1.mean()
mu2 = r2.mean()
pd.DataFrame({"μ1 (monthly)": mu1, "μ2 (monthly)": mu2,
              "μ1 (annualized)": mu1 * 12, "μ2 (annualized)": mu2 * 12}).round(4)

#### 3. Covariance Matrices

In [ ]:
sigma1 = r1.std()
sigma2 = r2.std()
cov1 = r1.cov()
cov2 = r2.cov()
print("Standard Deviation (Period 1):", sigma1)
print("Standard Deviation (Period 2):", sigma2)
print("Covariance (Period 1):")
display(cov1.round(5))
print("Covariance (Period 2):")
display(cov2.round(5))
print("Variance (Period 1):")
print(np.diag(cov1))
print("Variance (Period 2):")
print(np.diag(cov2))
cov_FSPTX_FIDSX = cov1.values[0,1]
print("Covariance between FSPTX and FIDSX (Period 1):", cov_FSPTX_FIDSX)
cov_FSPTX_FIDSX_2 = cov2.values[0,1]
print("Covariance between FSPTX and FIDSX (Period 2):", cov_FSPTX_FIDSX_2)
cov_FSPTX_FSCPX = cov1.values[0,2]
print("Covariance between FSPTX and FSCPX (Period 1):", cov_FSPTX_FSCPX)
cov_FSPTX_FSCPX_2 = cov2.values[0,2]
print("Covariance between FSPTX and FSCPX (Period 2):", cov_FSPTX_FSCPX_2)
cov_FIDSX_FSCPX = cov1.values[1,2]
print("Covariance between FIDSX and FSCPX (Period 1):", cov_FIDSX_FSCPX)
cov_FIDSX_FSCPX_2 = cov2.values[1,2]
print("Covariance between FIDSX and FSCPX (Period 2):", cov_FIDSX_FSCPX_2)

# Standard deviations and variances as column vectors
display(pd.DataFrame({"Std Dev (P1)": sigma1, "Std Dev (P2)": sigma2,
                      "Variance (P1)": np.diag(cov1), "Variance (P2)": np.diag(cov2)}).round(5))

# Pairwise covariances as column vectors
pairs = [("FSPTX", "FIDSX"), ("FSPTX", "FSCPX"), ("FIDSX", "FSCPX")]
display(pd.DataFrame({
    "Cov (P1)": [cov1.loc[a, b] for a, b in pairs],
    "Cov (P2)": [cov2.loc[a, b] for a, b in pairs],
}, index=[f"{a}, {b}" for a, b in pairs]).round(5))


#### 4. The Two Funds

In [ ]:
def two_funds(mu, Sigma):
    inv = np.linalg.inv(Sigma)
    ones = np.ones(len(mu))
    w0 = inv @ ones / (ones @ inv @ ones) #GMV portfolio weights
    w1 = inv @ mu - (ones @ inv @ mu) * w0
    return w0, w1

def mv_port(A, mu, sigma):
    w0, w1 = two_funds(mu, sigma)
    w = w0 + 1/A * w1
    return w

A_grid = np.arange(1, 101)
print(A_grid)

w0_1, w1_1 = two_funds(mu1.values, cov1.values)
w0_2, w1_2 = two_funds(mu2.values, cov2.values)

pd.DataFrame({"w0 (P1)": w0_1, "w1 (P1)": w1_1, "w0 (P2)": w0_2, "w1 (P2)": w1_2}, index=tickers_2)

#### 5. Utopian MVEF

In [ ]:
def port_stats(w, mu, sigma):
    w = np.asarray(w)
    return w @ mu, np.sqrt(w @ sigma @ w)

mu2v, cov2v = mu2.values, cov2.values

#Utopian Frontier on the A grid
W5 = np.array([mv_port(A, mu2v, cov2v) for A in A_grid])
mean5 = W5 @ mu2v
vol5 = np.sqrt(np.einsum('ij,jk,ik->i', W5, cov2v, W5))

#GMV and Sharpe (tangency) portfolios, computed exactly
mean_gmv, vol_gmv = port_stats(w0_2, mu2v, cov2v)
A_T = np.ones(3) @ np.linalg.inv(cov2v) @ mu2v
w_T = mv_port(A_T, mu2v, cov2v)
mean_T, vol_T = port_stats(w_T, mu2v, cov2v)

#Plot (annualized: mean x12, vol x sqrt(12), applied to every series)
k_m, k_v = 12, np.sqrt(12)
plt.figure(figsize=(7, 5))
plt.plot(vol5*k_v, mean5*k_m, label="Utopian MV frontier (A = 1..100)")
plt.scatter(vol_gmv*k_v, mean_gmv*k_m, color="green", s=60, zorder=3, label="GMV")
plt.scatter(vol_T*k_v, mean_T*k_m, color="red", marker="*", s=130, zorder=3, label="Sharpe")
plt.xlabel("Annualized volatility"); plt.ylabel("Annualized mean return")
plt.title("Mean-variance frontier using Period 2 estimates")
plt.grid(True); plt.legend()
plt.show()

# GMV and Sharpe portfolio summary (period 2, annualized)
pd.DataFrame({
    "GMV": list(w0_2) + [mean_gmv * k_m, vol_gmv * k_v, (mean_gmv * k_m) / (vol_gmv * k_v), np.nan],
    "Sharpe": list(w_T) + [mean_T * k_m, vol_T * k_v, (mean_T * k_m) / (vol_T * k_v), A_T],
}, index=["w_FSPTX", "w_FIDSX", "w_FSCPX", "Ann. mean", "Ann. vol", "Ann. Sharpe", "A"]).round(4)
    

#### 6. Realistic Out-of-Sample Evaluation

In [ ]:
mu1v, cov1v = mu1.values, cov1.values

#build portfolios with period 1 estimates, evaluate them with period 2 estimates
W6 = np.array([mv_port(A, mu1v, cov1v) for A in A_grid])
mean6 = W6 @ mu2v
vol6 = np.sqrt(np.einsum('ij,jk,ik->i', W6, cov2v, W6))

#period-1 GMV and Sharpe portfolios, evaluated with period 2 estimates
mean_gmv1, vol_gmv1 = port_stats(w0_1, mu2v, cov2v)
A_T1 = np.ones(3) @ np.linalg.inv(cov1v) @ mu1v
w_T1 = mv_port(A_T1, mu1v, cov1v)
mean_T1, vol_T1 = port_stats(w_T1, mu2v, cov2v)

plt.figure(figsize=(7, 5))
plt.plot(vol5 * k_v, mean5 * k_m, color="tab:blue", label="Utopian MV frontier (mu2, cov2)")
plt.plot(vol6 * k_v, mean6 * k_m, color="tab:orange", label="Out-of-sample MV frontier (mu1, cov1)")
plt.scatter(vol_gmv*k_v, mean_gmv*k_m, color="green", s=60, zorder=3, label="GMV")
plt.scatter(vol_T*k_v, mean_T*k_m, color="red", marker="*", s=130, zorder=3, label="Sharpe")
plt.scatter(vol_gmv1*k_v, mean_gmv1*k_m, color="purple", s=60, zorder=3, label="GMV (out-of-sample)")
plt.scatter(vol_T1*k_v, mean_T1*k_m, color="brown", marker="*", s=130, zorder=3, label="Sharpe (out-of-sample)")
plt.xlabel("Annualized volatility"); plt.ylabel("Annualized mean return")
plt.title("Utopian vs. Out-of-sample MV frontier")
plt.grid(True); plt.legend()
plt.show()

#### 7. Comparison of Utopian MV vs. Out-of-Sample MV

In [ ]:
inv2 = np.linalg.inv(cov2v); one = np.ones(3)
a, b, c = one @ inv2 @ one, one @ inv2 @ mu2v, mu2v @ inv2 @ mu2v
minvar = lambda mu: (a*mu**2 - 2*b*mu + c) / (a*c - b**2)

gap = vol6 - np.sqrt(minvar(mean6))
print("min gap (should be >= 0):", gap.min(), "| max gap:", gap.max())

#### 8. Parsimonious Build

In [ ]:
mu1v, cov1v = mu1.values, cov1.values
cov1_diag = np.diag(np.diag(cov1v))  # zero correlations

def frontier(A_grid, mu_est, S_est, mu_eval, S_eval):
    W = np.array([mv_port(A, mu_est, S_est) for A in A_grid])
    return W @ mu_eval, np.sqrt(np.einsum("ij,jk,ik->i", W, S_eval, W))

mean6, vol6 = frontier(A_grid, mu1v, cov1v, mu2v, cov2v)       # Part 6 (out-of-sample)
mean8, vol8 = frontier(A_grid, mu1v, cov1_diag, mu2v, cov2v)   # Part 8 (shrinkage)

# GMV and Sharpe portfolios for each set of estimates, all evaluated in period 2
def gmv_and_sharpe(mu_est, S_est):
    w0, _ = two_funds(mu_est, S_est)
    A_T = np.ones(3) @ np.linalg.inv(S_est) @ mu_est
    wT = mv_port(A_T, mu_est, S_est)
    return port_stats(w0, mu2v, cov2v), port_stats(wT, mu2v, cov2v)

(m_g1, v_g1), (m_T1, v_T1) = gmv_and_sharpe(mu1v, cov1v)
(m_gd, v_gd), (m_Td, v_Td) = gmv_and_sharpe(mu1v, cov1_diag)

plt.figure(figsize=(8, 6))
plt.plot(vol5 * k_v, mean5 * k_m, color="tab:blue", label="Utopian (mu2, cov2)")
plt.plot(vol6 * k_v, mean6 * k_m, color="tab:orange", label="Out-of-sample (mu1, cov1)")
plt.plot(vol8 * k_v, mean8 * k_m, color="tab:green", label="Shrinkage (mu1, diag cov1)")

plt.scatter(vol_gmv * k_v, mean_gmv * k_m, color="tab:blue", edgecolors="black", s=70, zorder=3, label="GMV (utopian)")
plt.scatter(vol_T * k_v, mean_T * k_m, color="tab:blue", edgecolors="black", marker="*", s=220, zorder=3, label="Sharpe (utopian)")
plt.scatter(v_g1 * k_v, m_g1 * k_m, color="tab:orange", edgecolors="black", s=70, zorder=3, label="GMV (out-of-sample)")
plt.scatter(v_T1 * k_v, m_T1 * k_m, color="tab:orange", edgecolors="black", marker="*", s=220, zorder=3, label="Sharpe (out-of-sample)")
plt.scatter(v_gd * k_v, m_gd * k_m, color="tab:green", edgecolors="black", s=70, zorder=3, label="GMV (shrinkage)")
plt.scatter(v_Td * k_v, m_Td * k_m, color="tab:green", edgecolors="black", marker="*", s=220, zorder=3, label="Sharpe (shrinkage)")

plt.xlabel("Annualized volatility"); plt.ylabel("Annualized mean return")
plt.title("Out-of-sample MV frontier with and without zero correlations")
plt.grid(True); plt.legend(fontsize=8)
plt.show()

# mean return of each frontier at the same volatility levels (annualized)
for v in [0.25, 0.30, 0.40, 0.50]:
    m_u = np.interp(v, vol5 * k_v, mean5 * k_m)
    m_6 = np.interp(v, vol6 * k_v, mean6 * k_m)
    m_8 = np.interp(v, vol8 * k_v, mean8 * k_m)
    print(f"Vol {v:.2f}: Utopian {m_u:.4f} | Out-of-sample {m_6:.4f} | Shrinkage {m_8:.4f}")

# GMV and Sharpe summary (evaluated in period 2, annualized)
pd.DataFrame({
    "Ann. mean": [mean_gmv, mean_T, m_g1, m_T1, m_gd, m_Td],
    "Ann. vol":  [vol_gmv, vol_T, v_g1, v_T1, v_gd, v_Td],
}, index=["GMV (utopian)", "Sharpe (utopian)", "GMV (out-of-sample)", "Sharpe (out-of-sample)",
          "GMV (shrinkage)", "Sharpe (shrinkage)"]).mul([k_m, k_v]).assign(
    **{"Ann. Sharpe": lambda d: d["Ann. mean"] / d["Ann. vol"]}).round(4)


### Risk Modeling

#### 1. Monthly Data and GBM Calibration

In [ ]:
spy = yf.download(tickers="SPY", start="1994-01-01", end="2026-10-01", interval="1mo", auto_adjust=False, progress=False)["Adj Close"]
s = spy.squeeze().dropna().astype(float)          # 393 monthly adj closes, Jan 1994 - Sep 2026
p = s.values
r = np.diff(np.log(p))                            # 392 monthly log returns
n, dt = len(r), 1/12
T = n*dt
S0, ST_act = p[0], p[-1]

sigma = r.std(ddof=1)/np.sqrt(dt)                 # annualized volatility
mu    = r.mean()/dt + sigma**2/2                  # annualized GBM drift
print(f"{len(s)} prices, {n} returns, {s.index[0]:%Y-%m} to {s.index[-1]:%Y-%m}")

pd.DataFrame({"Estimate": [mu, sigma]}, index=["μ (annual)", "σ (annual)"]).round(4)

#### 2. Monte Carlo Simulation

In [ ]:
rng = np.random.default_rng(535)
Z = rng.standard_normal((1000, n))

def simulate(mu, sigma, Z):
    inc = (mu - sigma**2/2)*dt + sigma*np.sqrt(dt)*Z
    return S0*np.exp(np.concatenate([np.zeros((Z.shape[0],1)), np.cumsum(inc, axis=1)], axis=1))

paths = simulate(mu, sigma, Z)                    # 1000 paths x 393 prices
ST = paths[:, -1]

pd.DataFrame({"Simulated": [ST.mean(), ST.var(ddof=1)],
              "True (GBM)": [S0*np.exp(mu*T), S0**2*np.exp(2*mu*T)*(np.exp(sigma**2*T)-1)]},
             index=["E[S_T | S_0]", "Var[S_T | S_0]"]).round(2)

#### 3. Closest Simulated Path

In [ ]:
d2 = np.linalg.norm(paths - p, axis=1)            # L2 distance of each path from the actual one
d2log = np.linalg.norm(np.log(paths) - np.log(p), axis=1)
best = d2.argmin()
print("closest (price):", best, d2.min(), "| closest (log):", d2log.argmin(), "| median dist:", np.median(d2))

plt.figure(figsize=(7, 3.8))
plt.plot(s.index, p, "k", lw=1.6, label="Actual SPY")
plt.plot(s.index, paths[best], lw=1.3, label=f"Closest simulated path (#{best+1})")
plt.legend(); plt.grid(True); plt.xlabel("Month"); plt.ylabel("Adjusted price (USD)")
plt.show()

#### 4. 99% VaR of the PNL

In [ ]:
var99 = lambda x: np.mean(x) - np.quantile(x, 0.01)
pnl = 100*(ST - S0)

pd.DataFrame({"Value": [pnl.mean(), np.quantile(pnl, .01), var99(pnl), -np.quantile(pnl, .01)]},
             index=["Mean PNL", "1st percentile PNL", "99% VaR (mean - q01)", "-q01"]).round(2)

#### 5. VaR as a Function of σ

##### (a)

In [ ]:
grid = np.round(np.arange(0.10, 0.5001, 0.01), 2)
sweep = []
for sg in grid:
    mu_s = r.mean()/dt + sg**2/2                  # recalibrate mu for each sigma
    sweep.append(var99(100*(simulate(mu_s, sg, Z)[:, -1] - S0)))
sweep = np.array(sweep)

z = stats.norm.ppf(0.01); mT = r.mean()*n
closed = 100*S0*(np.exp(mT + grid**2*T/2) - np.exp(mT + grid*np.sqrt(T)*z))

plt.figure(figsize=(7, 3.8))
plt.plot(grid, sweep/1e3, "o-", ms=3.5, label="Monte Carlo")
plt.plot(grid, closed/1e3, "--", label="Closed form")
plt.axvline(sigma, color="gray", ls=":", label=f"Calibrated σ = {sigma:.3f}")
plt.xlabel("σ"); plt.ylabel("99% VaR (USD thousands)"); plt.grid(True); plt.legend()
plt.show()

for sg in (0.10, 0.20, 0.30, 0.40, 0.50):
    i = list(grid).index(sg); print(sg, round(sweep[i]), round(closed[i]))

##### (b)

_Answer:_ 

##### (c)

_Answer:_ 

#### 6. Historical vs. Parametric VaR

##### (a)

In [ ]:
print("Historical 99% VaR:", var99(r))

##### (b)

In [ ]:
sim_r = np.diff(np.log(paths), axis=1).ravel()    # 1000 x 392 simulated monthly log returns, flattened
print("Parametric (MC) 99% VaR:", var99(sim_r))

##### (c)

In [ ]:
pd.DataFrame({
    "Historical": [var99(r), np.quantile(r, .01), r.min(), stats.skew(r), stats.kurtosis(r)],
    "Simulated GBM": [var99(sim_r), np.quantile(sim_r, .01), sim_r.min(), stats.skew(sim_r), stats.kurtosis(sim_r)],
}, index=["99% VaR", "1st percentile", "Min", "Skew", "Excess kurtosis"]).round(4)

_Answer:_ 

##### (d)

In [ ]:
xs = np.linspace(-0.3, 0.2, 500)
plt.figure(figsize=(7, 3.8))
plt.hist(r, bins=40, range=(-0.3, 0.2), density=True, alpha=.55, label="Historical")
plt.plot(xs, stats.gaussian_kde(sim_r)(xs), "r", label="Simulated GBM")
plt.xlabel("Monthly log return"); plt.legend(); plt.grid(True)
plt.show()

_Answer:_ 

##### (e)

_Answer:_ 

### Dynamic Games via Monte Carlo

#### 1. Two-Stage Problem

In [ ]:
HI = 10.0

def analytic(T):
    U = np.zeros(T + 2); U[1] = HI / 2
    for k in range(1, T + 1):
        U[k + 1] = U[k]**2 / (2*HI) + HI/2          # E[max(X, U_k)]
    thr = np.array([U[T - t] if t < T else 0.0 for t in range(T + 1)])
    return thr, U[T + 1]

def threshold_from_fit(beta):
    a, b, c = beta
    xs = np.linspace(0, HI, 20001); s_ = xs - (a + b*xs + c*xs**2)
    i = np.where((s_[:-1] <= 0) & (s_[1:] > 0))[0]
    if len(i) == 0: return HI if s_[-1] <= 0 else 0.0
    i = i[0]; return xs[i] - s_[i]*(xs[i+1] - xs[i])/(s_[i+1] - s_[i])

def lsmc(T, N=1_000_000, seed=2026):
    rng = np.random.default_rng(seed)
    CF = rng.uniform(0, HI, N)                       # stage T: forced stop, payoff = X_T
    coefs, thr = {}, np.zeros(T + 1)
    for t in range(T - 1, -1, -1):                   # backward induction
        X = rng.uniform(0, HI, N)
        B = np.vander(X, 3, increasing=True)         # basis [1, x, x^2]
        beta = np.linalg.lstsq(B, CF, rcond=None)[0] # continuation value C_t(x)
        CF = np.where(X > B @ beta, X, CF)           # stop if X_t > C_t(X_t)
        coefs[t] = beta; thr[t] = threshold_from_fit(beta)
    return coefs, thr, CF.mean(), CF.std(ddof=1)/np.sqrt(N)

def evaluate(T, coefs, N=4_000_000, seed=2027):      # fresh paths, fixed rules (no in-sample bias)
    rng = np.random.default_rng(seed)
    CF = rng.uniform(0, HI, N)
    for t in range(T - 1, -1, -1):
        X = rng.uniform(0, HI, N)
        CF = np.where(X > np.vander(X, 3, increasing=True) @ coefs[t], X, CF)
    return CF.mean(), CF.std(ddof=1)/np.sqrt(N)

def run(T):
    coefs, thr, v_in, se_in = lsmc(T)
    v_out, se_out = evaluate(T, coefs)
    thr_true, v_true = analytic(T); thr[T] = 0.0
    print(f"LSMC value in-sample {v_in:.4f} (SE {se_in:.4f}) | out-of-sample {v_out:.4f} (SE {se_out:.4f}) | analytic {v_true:.4f}")
    print("value of waiting vs taking first draw (value - 5):", round(v_true - 5, 3))
    table = pd.DataFrame({"a": [coefs[t][0] for t in range(T)], "b": [coefs[t][1] for t in range(T)],
                          "c": [coefs[t][2] for t in range(T)],
                          "LSMC threshold": thr[:T], "Exact threshold": thr_true[:T]}, index=range(T))
    table.index.name = "t"
    return thr, thr_true, v_in, v_out, v_true, table

res = {}
res[2] = run(2)
res[2][-1].round(4)

#### 2. Ten-Stage Problem

In [ ]:
res[10] = run(10)
thr, thr_true = res[10][0], res[10][1]

plt.figure(figsize=(7, 3.8))
plt.plot(range(11), thr, "o-", label="LSMC threshold")
plt.plot(range(11), thr_true, "--", label="Exact threshold")
plt.xlabel("Stage t"); plt.ylabel("Stop if X_t above"); plt.title("Stopping threshold, 10-stage game")
plt.grid(True); plt.legend()
plt.show()

res[10][-1].round(4)

#### 3. One-Hundred-Stage Problem

In [ ]:
res[100] = run(100)
thr, thr_true = res[100][0], res[100][1]
print("thresholds t=0,10,50,80,99:", [(t, round(thr[t], 3), round(thr_true[t], 3)) for t in (0, 10, 50, 80, 99)])
print("max |LSMC - exact| over t<T:", np.abs(thr[:100] - thr_true[:100]).max().round(3))

plt.figure(figsize=(7, 3.8))
plt.plot(range(101), thr, label="LSMC threshold")
plt.plot(range(101), thr_true, "--", label="Exact threshold")
plt.xlabel("Stage t"); plt.ylabel("Stop if X_t above"); plt.title("Stopping threshold, 100-stage game")
plt.grid(True); plt.legend()
plt.show()

pd.DataFrame({"LSMC (in-sample)": [res[T][2] for T in (2, 10, 100)],
              "LSMC (out-of-sample)": [res[T][3] for T in (2, 10, 100)],
              "Analytic": [res[T][4] for T in (2, 10, 100)],
              "Threshold at t=0": [res[T][0][0] for T in (2, 10, 100)]},
             index=["2-stage", "10-stage", "100-stage"]).round(4)

#### 4. Option to Wait

The values of options added are diminishing the more step you go which make sense because early on there are more possible chances and it keeps shrinking as the steps go on. The option greek Theta agrees with this.